# Ablacion HSG18 fed pilot — `lr_backbone=1e-4`

Ablacion diagnostica para distinguir si el resultado pobre de
`hsg18_fedavg_pilot_full_finetuning_lr1e-5` se debe a:

- **A)** problema de adaptacion downstream por LR demasiado conservador sobre el ckpt FL (10x menos informativo que central) → `lr_backbone=1e-4` deberia mejorar;
- **B)** problema estructural del embedding FL en dominio HDD (cliente `hdd` con 1 dataset HSF15) → ningun LR razonable arreglara HSG18.

Resultados conocidos hasta ahora (macro_f1 test):

```
HSG18  from_scratch          = 0.5693
HSG18  central_linear        = 0.9056
HSG18  central_full_lr1e-5   = 0.9504
HSG18  fed_linear            = 0.6080
HSG18  fed_full_lr1e-5       = 0.5547   <- colapsa a clase mayoritaria
                                            recall clase 0 = 24.2 %
                                            recall clase 1 = 99.3 %
```

**Restricciones**:
- NO lanzar full FL.
- NO implementar FedProx.
- NO modificar sampling policy ni `min_client_presence`.
- NO repetir CWRU, ni linear, ni lr1e-5.
- Solo 1 corrida nueva sobre HSG18 con `lr_backbone=1e-4` y el mismo `ckpt_final.pt` FL pilot.
- Coste estimado: ~15 min en A100.

**Criterio interpretativo (NO decide full FL)**:
- Si `macro_f1_lr1e-4 > macro_f1_lr1e-5` Y `recall clase 0` mejora sin destruir clase 1 → A (parcialmente adaptacion).
- Si `macro_f1_lr1e-4 <= fed_linear` o colapsa de otra forma → B (estructural).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. colab_init

In [ ]:
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 3. Pull al HEAD

Debe incluir el commit de la ablacion (config + test + este notebook).

In [ ]:
%cd /content/fm_fl_phmd
!git pull --ff-only
!git log -3 --oneline
!git status --porcelain

## 4. Verificar GPU

**Si NO es A100, para aqui**.

In [ ]:
!nvidia-smi | head -15

## 5. pytest preflight

Incluye `test_ablacion_hsg18_lr1e4_consistente_con_base` que blinda el nuevo config bit-a-bit contra la version 1e-5.

In [ ]:
!python -m pytest \
  tests/test_downstream_federated_pilot_configs.py \
  tests/test_downstream_metrics.py \
  tests/test_downstream_pooling.py \
  tests/test_downstream_adaptive_batch.py -q

## 6. Verificar el checkpoint federado

Mismo `ckpt_final.pt` FL pilot que en las 4 corridas anteriores. NO usar `ckpt_round005.pt` ni `ckpt_round010.pt`.

In [ ]:
import torch
from pathlib import Path

CKPT_FL = Path(
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt"
)

!ls -lh "{CKPT_FL}"

ck = torch.load(str(CKPT_FL), map_location="cpu", weights_only=False)
assert "model_state_dict" in ck, "ckpt FL no tiene model_state_dict"
print("keys top-level:", list(ck.keys()))
sd = ck["model_state_dict"]
print(f"model_state_dict: {len(sd)} tensores, primer key: {next(iter(sd))}")
for k in ("config_hash", "git_hash", "stage", "run_name", "round", "param_count"):
    if k in ck:
        print(f"{k}: {ck[k]}")

## 7. Preparar `_stdout/`

In [ ]:
!mkdir -p /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/hsg18/_stdout

## 8. Dry-run de la ablacion

Sanity rapido: manifest HSG18 leido, n_channels=1, effective_bc=64, checkpoint cargado, forward sintetico OK, n_trainable_params ~802 066.

In [ ]:
CKPT_FL = (
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt"
)

!python -m training.train_downstream_classification \
  --config training/configs/downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-4.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_FL}" \
  --dry-run 2>&1 | tail -20

## 9. Corrida real unica

~15 min en A100 (HSG18 con C=1, 38 896 ventanas).

In [ ]:
import time

CKPT_FL = (
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt"
)

t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-4.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_FL}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/hsg18/_stdout/downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-4.stdout.log
print(f"\n[total] HSG18 fed pilot full_ft lr1e-4 elapsed = {time.time() - t0:.1f} s")

## 10. Summary comparativa

Compara el nuevo run lr1e-4 contra:
- el run lr1e-5 (colapso a clase mayoritaria),
- el run linear_probing,
- los baselines `from_scratch` y central.

In [ ]:
import json
from pathlib import Path

base = Path("/content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/hsg18")
paths = {
    "fed_linear":       base / "downstream_hsg18_fedavg_pilot_linear_probing"            / "run_info.json",
    "fed_full_lr1e-5":  base / "downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-5"    / "run_info.json",
    "fed_full_lr1e-4":  base / "downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-4"    / "run_info.json",
}
rows = {}
for name, p in paths.items():
    if not p.is_file():
        print(f"WARN: no encontrado {p}")
        continue
    ri = json.loads(p.read_text())
    tm = ri.get("test_metrics") or {}
    rows[name] = {
        "macro_f1":             tm.get("macro_f1", 0.0),
        "balanced_accuracy":    tm.get("balanced_accuracy", 0.0),
        "accuracy":             tm.get("accuracy", 0.0),
        "best_epoch":           ri.get("best_epoch", -1),
        "elapsed_seconds":      ri.get("elapsed_seconds", 0.0),
        "n_trainable_params":   ri.get("n_trainable_params", 0),
        "config_hash":          ri.get("config_hash", ""),
        "amp_nonfinite_grad_steps": ri.get("amp_nonfinite_grad_steps", -1),
        "confusion_matrix":     tm.get("confusion_matrix"),
        "per_class":            tm.get("per_class"),
    }

print(f"{'mode':<22s} {'macro_f1':>10s} {'bal_acc':>10s} {'acc':>8s} {'best_ep':>8s} {'elapsed':>10s} {'n_train':>10s} {'amp_nf':>7s} {'config_hash':>20s}")
print("-" * 130)
for name, r in rows.items():
    print(f"{name:<22s} {r['macro_f1']:>10.4f} {r['balanced_accuracy']:>10.4f} {r['accuracy']:>8.4f} {r['best_epoch']:>8d} {r['elapsed_seconds']:>10.1f}s {r['n_trainable_params']:>10d} {r['amp_nonfinite_grad_steps']:>7d} {r['config_hash']:>20s}")

# Confusion + per_class del nuevo run
if "fed_full_lr1e-4" in rows:
    r = rows["fed_full_lr1e-4"]
    print("\n=== fed_full_lr1e-4: confusion_matrix ===")
    print(json.dumps(r["confusion_matrix"], indent=2))
    print("\n=== fed_full_lr1e-4: per_class ===")
    print(json.dumps(r["per_class"], indent=2))

# Deltas
baselines = {
    "from_scratch":           0.5693,
    "central_linear":         0.9056,
    "central_full_lr1e-5":    0.9504,
}
if "fed_full_lr1e-4" in rows and "fed_full_lr1e-5" in rows and "fed_linear" in rows:
    f_lr4 = rows["fed_full_lr1e-4"]["macro_f1"]
    f_lr5 = rows["fed_full_lr1e-5"]["macro_f1"]
    f_lin = rows["fed_linear"]["macro_f1"]
    print("\n=== Deltas macro_f1 (nuevo run lr1e-4 vs referencias) ===")
    print(f"  vs fed_full_lr1e-5    : {f_lr4 - f_lr5:+.4f}")
    print(f"  vs fed_linear         : {f_lr4 - f_lin:+.4f}")
    print(f"  vs from_scratch       : {f_lr4 - baselines['from_scratch']:+.4f}")
    print(f"  vs central_full_lr1e-5: {f_lr4 - baselines['central_full_lr1e-5']:+.4f}")
    ratio = f_lr4 / baselines['central_full_lr1e-5']
    print(f"  ratio fed_lr1e-4 / central_full = {ratio:.4f}  ({ratio*100:.1f} %)")

    # Recalls clase 0 y 1 del nuevo run
    pc = rows["fed_full_lr1e-4"].get("per_class") or {}
    rec = pc.get("recall") or []
    if len(rec) >= 2:
        print(f"\n  recall clase 0 (lr1e-4): {rec[0]:.4f}")
        print(f"  recall clase 1 (lr1e-4): {rec[1]:.4f}")
        # Comparacion con lr1e-5 (recall clase 0 = 24.2%, clase 1 = 99.3%)
        print(f"  recall clase 0 (lr1e-5): 0.2417  (referencia)")
        print(f"  recall clase 1 (lr1e-5): 0.9934  (referencia)")

    # Criterio interpretativo
    print("\n=== Lectura ===")
    if f_lr4 > f_lr5 and len(rec) >= 2 and rec[0] > 0.30 and rec[1] > 0.70:
        print("  Hipotesis A favorecida: lr1e-5 era demasiado conservador. lr1e-4 escapa el colapso.")
    elif f_lr4 <= f_lin or (len(rec) >= 2 and (rec[0] < 0.10 or rec[1] < 0.50)):
        print("  Hipotesis B favorecida: embedding FL HDD insuficiente, ningun LR razonable lo arregla.")
    else:
        print("  Mixto: lr1e-4 mejora algo pero no resuelve completamente. Reportar con cautela.")

print("\nPega este output completo en el chat para cerrar la ablacion HSG18.")
print("NO autorizar full FL solo con este resultado.")